In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import shutil
import os

# paths
originalPath = r"data\Dataset_ATS_v2.csv"
backupPath = r"data\Dataset_ATS_v2_backup.csv"

# take a backup first so the raw file is never lost
if not os.path.exists(backupPath):
    shutil.copyfile(originalPath, backupPath)
    print("backup created at " + backupPath)
else:
    print("backup already exists at " + backupPath)

df = pd.read_csv(backupPath)
print("dataset loaded, shape ->", df.shape)

# fill missing values - median for numeric, mode for categorical
numericalCols = ["tenure", "MonthlyCharges"]
categoricalCols = ["Dependents", "PhoneService", "MultipleLines",
                   "gender", "InternetService", "Contract", "Churn"]

for col in numericalCols:
    df[col]= df[col].fillna(df[col].median())

for col in categoricalCols:
    df[col] = df[col].fillna(df[col].mode()[0])

# encode yes/no + gender columns to 1/0
binaryCols = ["Dependents", "PhoneService", "MultipleLines", "Churn"]
for col in binaryCols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

df["gender"] = df["gender"].map({"Male": 1, "Female": 0})
df = pd.get_dummies(df, columns=["InternetService", "Contract"], drop_first=True)

# bucket tenure into short/medium/long
def getTenureGroup(tenure):
    if tenure <= 12:
        return "Short"
    elif tenure <= 48:
        return "Medium"
    else:
        return "Long"

df["tenure_group"] = df["tenure"].apply(getTenureGroup)
df = pd.get_dummies(df, columns=["tenure_group"], drop_first=True)

# flag customers paying above the average monthly charge
avgCharge = df["MonthlyCharges"].mean()
df["HighMonthlyCharge"] = df["MonthlyCharges"].apply(lambda x: 1 if x > avgCharge else 0)

# interaction feature - tenure x monthly charge
df["tenure_x_charge"] = df["tenure"] * df["MonthlyCharges"]

# scale the numeric + engineered columns
scaler = StandardScaler()
scaleCols = ["tenure", "MonthlyCharges", "tenure_x_charge"]
df[scaleCols] = scaler.fit_transform(df[scaleCols])

# drop_duplicates()
df = df.drop_duplicates()

df.to_csv(originalPath, index=False)
print("cleaned dataset saved, backup preserved at " + backupPath)

backup already exists at data\Dataset_ATS_v2_backup.csv
dataset loaded, shape -> (7043, 10)
cleaned dataset saved, backup preserved at data\Dataset_ATS_v2_backup.csv
